In [4]:
!pip install typesafe-sdk

In [6]:
from typesafe_sdk import Choice, TypeSafeClient
from dotenv import load_dotenv
load_dotenv(".env")

client = TypeSafeClient()

In [9]:
import json 

with open("./ds1_test.json", "r") as f:
    data = json.load(f)

In [ ]:
## context -- list, outcome -- string, meta(family, expected)
# context is like IDs we need to fetch the item names from the ds2 and vice versa.

for d in data:
    print(d.items())
    break

dict_items([('context', ['OB_1001', 'OB_1014']), ('outcome', 'OB_1015'), ('meta', {'family': 'bread+milk->butter', 'kind': 'seen', 'expected': 'OB_1015', 'combo': 'OB_1001|OB_1014'})])


In [37]:
## Setting up the base apriori 
from apriori_core import load_json, mine, predict
train = load_json("ds1_train.json")

# Mine Apriori rules once
rules, n_itemsets = mine(train, min_support=0.002, min_conf=0.1)


In [87]:
import json
import pandas as pd

from apriori_core import mine, predict

with open("ds1_test.json", "r") as f:
    data_test = json.load(f)

with open("ds2_descriptions.json", "r") as f:
    descriptions = json.load(f)

# 2. Create product lookup dictionary
look_up_dict = {
    item["item_id"]: (
        item["item_id"],
        item["name"],
        item["labels"]
    )
    for item in descriptions
}

print(f"Mined {len(rules)} rules")

# 4. Run vanilla Apriori on the same 400 test cases
apriori_results = []

for i, test_row in enumerate(data_test):
    cart = test_row["context"]

    # Top-1 recommendation by Apriori confidence
    predictions = predict(rules, cart, k=1)

    if predictions:
        recommended_id = predictions[0][0]
        recommended_name = look_up_dict[recommended_id][1]
    else:
        recommended_name = None

    cart_names = [
        look_up_dict[item_id][1]
        for item_id in cart
    ]

    apriori_results.append({
        "cart": ", ".join(cart_names),
        "outcome": recommended_name
    })

    print(f"Done {i + 1}/{min(400, len(data_test))}")

# 5. Save in exactly the same format as JEV
df = pd.DataFrame(apriori_results)

df.to_csv("apriori_results.csv", index=False)

print("Saved apriori_results.csv")
print(df.head())

Mined 213 rules
Done 1/400
Done 2/400
Done 3/400
Done 4/400
Done 5/400
Done 6/400
Done 7/400
Done 8/400
Done 9/400
Done 10/400
Done 11/400
Done 12/400
Done 13/400
Done 14/400
Done 15/400
Done 16/400
Done 17/400
Done 18/400
Done 19/400
Done 20/400
Done 21/400
Done 22/400
Done 23/400
Done 24/400
Done 25/400
Done 26/400
Done 27/400
Done 28/400
Done 29/400
Done 30/400
Done 31/400
Done 32/400
Done 33/400
Done 34/400
Done 35/400
Done 36/400
Done 37/400
Done 38/400
Done 39/400
Done 40/400
Done 41/400
Done 42/400
Done 43/400
Done 44/400
Done 45/400
Done 46/400
Done 47/400
Done 48/400
Done 49/400
Done 50/400
Done 51/400
Done 52/400
Done 53/400
Done 54/400
Done 55/400
Done 56/400
Done 57/400
Done 58/400
Done 59/400
Done 60/400
Done 61/400
Done 62/400
Done 63/400
Done 64/400
Done 65/400
Done 66/400
Done 67/400
Done 68/400
Done 69/400
Done 70/400
Done 71/400
Done 72/400
Done 73/400
Done 74/400
Done 75/400
Done 76/400
Done 77/400
Done 78/400
Done 79/400
Done 80/400
Done 81/400
Done 82/400
Done 83/4

In [16]:
import json 

with open("./ds1_test.json", "r") as f:
    data_test = json.load(f)

with open("./ds2_descriptions.json", "r") as f:
    lookup = json.load(f)

## How we need to post the data to jev is as follows
### State: from the provided criteria, determine which item should be suggested as next for the user.
### questions : Department : Choice : instruction "Choose the best item that fits the cart : obj_id | item.name | labels"
### Criteria: outcome of apriori with labels. outcomte : label 

In [42]:
## Preprocessing for the labels and the data to be tested
look_up_dict = {}
for obj in lookup:
    look_up_dict[obj["item_id"]] = [obj["item_id"],obj["name"], " | ".join(obj["labels"])]
    ## lookup_dict [0] is name [1] is labels and the key is the obj id

In [65]:
## Now we process the input loop for the 400 test cases we have
client = TypeSafeClient()
cart_response = list()
for test_item in range(len(data_test)):
    ## get the candidates
    cart = data_test[test_item]["context"]
    candidates = [
        item_id
        for item_id, confidence, support, antecedent
        in predict(rules, cart, k=4)
    ]
    state = "from the provided criteria, determine which item should be suggested as next for the buyer."
    instruction = "Choose the best item that fits the cart : "+ "\n".join(f"{obj_id} | {obj_name} | {''.join(labels)}"for obj_id, obj_name, labels in [look_up_dict[item_id] for item_id in cart])
    criteria = {
        f"{obj_id} | {obj_name}": obj_labels
        for obj_id, obj_name, obj_labels in
        [look_up_dict[item_id] for item_id in candidates]
    }
    response = client.system_one(
        state=state,
        questions={
            "department": Choice(
                instructions=instruction,
                criteria=criteria,
            ),
        },
    )
    
    cart_response.append([cart, response.answers["department"].choice])
    print(f"Done for {test_item} out of {len(data_test)}")
    
    

Done for 0 out of 400
Done for 1 out of 400
Done for 2 out of 400
Done for 3 out of 400
Done for 4 out of 400
Done for 5 out of 400
Done for 6 out of 400
Done for 7 out of 400
Done for 8 out of 400
Done for 9 out of 400
Done for 10 out of 400
Done for 11 out of 400
Done for 12 out of 400
Done for 13 out of 400
Done for 14 out of 400
Done for 15 out of 400
Done for 16 out of 400
Done for 17 out of 400
Done for 18 out of 400
Done for 19 out of 400
Done for 20 out of 400
Done for 21 out of 400
Done for 22 out of 400
Done for 23 out of 400
Done for 24 out of 400
Done for 25 out of 400
Done for 26 out of 400
Done for 27 out of 400
Done for 28 out of 400
Done for 29 out of 400
Done for 30 out of 400
Done for 31 out of 400
Done for 32 out of 400
Done for 33 out of 400
Done for 34 out of 400
Done for 35 out of 400
Done for 36 out of 400
Done for 37 out of 400
Done for 38 out of 400
Done for 39 out of 400
Done for 40 out of 400
Done for 41 out of 400
Done for 42 out of 400
Done for 43 out of 40

In [72]:
import pandas as pd

df = pd.DataFrame(
    [
        {
            "cart": ", ".join(cart),
            "recommendation": recommendation
        }
        for cart, recommendation in cart_response
    ]
)

df.to_csv("recommendations.csv", index=False)

In [81]:
from apriori_core import load_json, mine, predict

# Load training data
train = load_json("ds1_train.json")

# Mine Apriori rules once
rules, n_itemsets = mine(train, min_support=0.004, min_conf=0.3)

# Input shopping cart
cart = ["OB_1078", "OB_1076", "OB_1002", "OB_1057"]  # Vegan Sausages + Sourdough

# Find matching Apriori recommendations
recommendations = predict(rules, cart, k=3)

for item_id, confidence, support, antecedent in recommendations:
    print(f"Product: {item_id}")
    print(f"Confidence: {confidence:.3f}")
    print(f"Support: {support:.3f}")
    print(f"Matched rule: {set(antecedent)} -> {item_id}")
    print()

Product: OB_1060
Confidence: 0.796
Support: 0.022
Matched rule: {'OB_1057'} -> OB_1060

Product: OB_1061
Confidence: 0.361
Support: 0.017
Matched rule: {'OB_1002'} -> OB_1061

Product: OB_1016
Confidence: 0.325
Support: 0.015
Matched rule: {'OB_1002'} -> OB_1016



In [82]:
converted_results = [
    {
        "cart": [look_up_dict[item_id][1] for item_id in cart],
        "outcome": look_up_dict[outcome.split(" | ")[0]][1]
    }
    for cart, outcome in cart_response
]

for row in converted_results:
    print(f"{', '.join(row['cart'])} => {row['outcome']}")

Whole Milk, Rye Bread => Vegan Butter
Bran Loaf, Almond Milk => Salted Butter
Vegan Burger Patties, Vegan Hot Dog Buns => Vegan Sausages
Meat-Free Burger Patties, Vegan Hot Dog Buns, Bin Bags => Vegan Sausages
Vegan Ham Slices, Brioche Bun => Vegan Mayo
Vegan Burger Patties, Hot Dog Buns => Pork Sausages
Ham Slices, Brioche Bun => Mayonnaise
Vegan Pesto, Gluten-Free Penne, Bananas => Vegan Parmesan
Vegan Cheese Slices, Bananas, Sourdough => Vegan Mayo
Rye Bread, Skimmed Milk => Vegan Butter
Gluten-Free Muesli, Almond Milk => Coconut Yogurt
Eggless Bun, Vegan Ham Slices => Vegan Mayo
White Bread, Vegan Cheese Slices, Kitchen Roll => Vegan Mayo
Sourdough, Soy Milk => Vegan Mayo
Corn Flakes, Lactose-Free Milk, Basmati Rice => Lactose-Free Yogurt
Bananas, Brioche Hot Dog Buns, Vegan Garlic Sauce, Dishwasher Tablets => Vegan Sausages
Cheddar Slices, White Bread => Salted Butter
Skimmed Milk, Plain Flour => Low Fat Yogurt
Vegan Bolognese Sauce, Egg Tagliatelle, Dishwasher Tablets => Vegetari

In [86]:
import pandas as pd

df = pd.DataFrame(converted_results)

# Convert cart lists into readable strings
df["cart"] = df["cart"].apply(lambda items: ", ".join(items))

# Save to CSV
df.to_csv("jev_results.csv", index=False)

print("Jev_only_results.csv")

Jev_only_results.csv


In [83]:
client = TypeSafeClient()
cart_response = []

for test_item in range(min(100, len(data_test))):
    row = data_test[test_item]
    cart = row["context"]
    expected = row["outcome"]

    apriori_results = predict(rules, cart, k=4)
    candidates = [item_id for item_id, *_ in apriori_results]

    if not candidates:
        cart_response.append({
            "cart": cart,
            "expected": expected,
            "candidates": [],
            "apriori_top1": None,
            "jev_choice": None,
            "status": "no_candidates"
        })
        continue

    cart_description = "\n".join(
        f"- {look_up_dict[item_id][0]} | "
        f"{look_up_dict[item_id][1]} | "
        f"{', '.join(look_up_dict[item_id][2])}"
        for item_id in cart
    )

    criteria = {
        f"{obj_id} | {obj_name}": obj_labels
        for obj_id, obj_name, obj_labels in
        (look_up_dict[item_id] for item_id in candidates)
    }

    state = (
        "Recommend one grocery product from the provided candidates. "
        "Consider meal compatibility and explicit dietary labels. "
        "Do not assume that buying a plant-based product proves "
        "the customer is vegan."
    )

    instruction = (
        "Choose the most suitable candidate for this cart.\n"
        "Prioritize explicit dietary compatibility, then meal relevance.\n"
        "Select only from the supplied candidates.\n\n"
        f"Cart:\n{cart_description}"
    )

    response = client.system_one(
        state=state,
        questions={
            "department": Choice(
                instructions=instruction,
                criteria=criteria,
            ),
        },
    )

    selected = response.answers["department"].choice
    selected_id = selected.split(" | ")[0]

    cart_response.append({
        "cart": cart,
        "expected": expected,
        "candidates": candidates,
        "apriori_top1": candidates[0],
        "jev_choice": selected_id,
        "valid_choice": selected_id in candidates,
        "expected_in_candidates": expected in candidates,
        "apriori_correct": candidates[0] == expected,
        "jev_correct": selected_id == expected
    })

In [84]:
cart_response


[{'cart': ['OB_1001', 'OB_1014'],
  'expected': 'OB_1015',
  'candidates': ['OB_1015', 'OB_1018', 'OB_1059'],
  'apriori_top1': 'OB_1015',
  'jev_choice': 'OB_1018',
  'valid_choice': True,
  'expected_in_candidates': True,
  'apriori_correct': True,
  'jev_correct': False},
 {'cart': ['OB_1011', 'OB_1007'],
  'expected': 'OB_1015',
  'candidates': ['OB_1015'],
  'apriori_top1': 'OB_1015',
  'jev_choice': 'OB_1015',
  'valid_choice': True,
  'expected_in_candidates': True,
  'apriori_correct': True,
  'jev_correct': True},
 {'cart': ['OB_1072', 'OB_1064'],
  'expected': 'OB_1075',
  'candidates': ['OB_1073', 'OB_1075'],
  'apriori_top1': 'OB_1073',
  'jev_choice': 'OB_1075',
  'valid_choice': True,
  'expected_in_candidates': True,
  'apriori_correct': False,
  'jev_correct': True},
 {'cart': ['OB_1071', 'OB_1064', 'OB_1080'],
  'expected': 'OB_1074',
  'candidates': ['OB_1074', 'OB_1073', 'OB_1075'],
  'apriori_top1': 'OB_1074',
  'jev_choice': 'OB_1075',
  'valid_choice': True,
  'ex

In [85]:
import pandas as pd

df = pd.DataFrame(cart_response)

print("Total test cases:", len(df))
print("Candidate recall@4:", df["expected_in_candidates"].mean())
print("Apriori accuracy:", df["apriori_correct"].mean())
print("JEV accuracy:", df["jev_correct"].mean())
print("Valid JEV selection:", df["valid_choice"].mean())

df.to_csv("jev_vs_apriori.csv", index=False)

Total test cases: 100
Candidate recall@4: 0.88
Apriori accuracy: 0.6
JEV accuracy: 0.5
Valid JEV selection: 1.0
